# 09 - Group Fitting


This notebook fits one shared simulation mechanism across multiple measured CVs. It first fits one Ar CV, then uses an Ar scan-rate group fit to refine shared `EE` potentials and tied diffusion, and finally fits a CO2/PhOH concentration series with an EEC' mechanism.


## Import eCAT And Set Paths

Use this notebook after the simulation intro, when the single-CV mechanism and parameter conventions are familiar.


In [1]:
from copy import deepcopy
import importlib.util
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name

import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
HAS_ELECTROKITTY = importlib.util.find_spec("electrokitty") is not None
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))
print("ElectroKitty available:", HAS_ELECTROKITTY)
if not HAS_ELECTROKITTY:
    print('Install in this notebook with: %pip install "ecat[simulation]"; then restart the kernel before running fit cells.')

eCAT version: 0.1.0b3
Example data: examples/data/fe_phoh_cv
Text files: 13
ElectroKitty available: False
Install in this notebook with: %pip install "ecat[simulation]"; then restart the kernel before running fit cells.


## Load And Select Candidate Traces

The group fits use public eCAT filters so the selected traces are reproducible. PhOH CVs are scaled against one shared Ar/Fc segment-2 wave before any fitting input is trimmed. Ar CVs use the `0` to `-1.5 V` fitting window; PhOH CVs use expanded waveform trimming from `-1` to `-1.7 V` during `cv_data()` conversion.


In [2]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "reference mode": "keyword",
    "reference keyword": "Fc",
    "reference guess": 0.4,
    "reference label": "Fc/Fc+",
    "electrode diameter": 0.3,
    "print": False,
})
cvs = e.filter(cvs, {"segments": 3}, {"print": False})

fe_ar = e.filter(cvs, {
    "gas": "Ar",
    "compounds": "Fe-tpyPY2Me",
}, {"logic": "AND", "print": False})

phoh_co2 = e.filter(cvs, {
    "gas": "CO2",
    "compounds": "PhOH",
}, {"logic": "AND", "print": False})

scan_series = e.filter(fe_ar, {
    "scan window": [-1.7, 1],
}, {"print": False})
scan_series = e.sort(scan_series, "scan rate", {"print": True})

ar_reference_cv = e.filter(scan_series, {"scan rate": 0.1}, {"print": False})[0]
phoh_group = e.filter(phoh_co2, {"scan rate": 0.1}, {"print": True})
phoh_28m_cv = e.filter(phoh_group, {"species": "2.8M PhOH"}, {"print": False})[0]
AR_CV_POTENTIAL_WINDOW = [-0.7, -1.7]
PHOH_CV_POTENTIAL_WINDOW = [-1.0, -1.7]
AR_CV_WINDOW_OPTIONS = {
    "potential window": AR_CV_POTENTIAL_WINDOW,
    "trim mode": "expand",
}
PHOH_CV_WINDOW_OPTIONS = {
    "potential window": PHOH_CV_POTENTIAL_WINDOW,
    "trim mode": "expand",
}


Searching recursively through:
 examples/data/fe_phoh_cv
13 .txt files found.



Reference correction:
  Mode: keyword
  Keyword: Fc
  Guess: 0.4 V
  Folder reference: MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_to_1V_100mVs.txt = 0.4665 V
  Usage:
    folder/ancestor reference: 3
    self-referenced successfully: 10


=== Sorted Objects ===
[Conditions] Exp Type: CV, Solvent: MeCN, Gas: Ar, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.7, 1], Segments: 3, IR Comp Percent: 100 %


,Scan Rate
[0],25 mV/s
[1],50 mV/s
[2],100 mV/s
[3],500 mV/s
[4],1 V/s


=== Filtered Objects (Include) ===
Filtering Criteria: (scan rate: 0.1)
Matched Objects: 4 / 4
[Conditions] Exp Type: CV, Solvent: MeCN, Gas: CO₂, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.2, 1], Scan Rate: 100 mV/s, Segments: 3, IR Comp Percent: 100 %


,Compounds
[0],100 mM PhOH
[1],560 mM PhOH
[2],1 M PhOH
[3],2.8 M PhOH


## Scale PhOH CVs Against One Ar/Fc Wave

Use the same Ar trace for every CO2/PhOH CV. The Fc wave is measured on segment 2 before trimming, then `scale_current()` applies that current scale to raw PhOH current copies. These scaled CV copies are what enter the EEC' group fit.

In [3]:
FC_SCALE_OPTIONS = {
    "reference cv": ar_reference_cv,
    "reference mode": "single",
    "segment": 2,
    "guess potential": 0.0,
    "print": True,
}

phoh_group_scaled = e.scale_current(phoh_group, FC_SCALE_OPTIONS)
phoh_28m_cv_scaled = e.filter(phoh_group_scaled, {"species": "2.8M PhOH"}, {"print": False})[0]

e.multiplot(phoh_group_scaled, {'print': False})

pd.DataFrame({
    "Label": [cv.name for cv in phoh_group_scaled],
    "Scale Factor": [getattr(cv, "current_scale_factor", None) for cv in phoh_group_scaled],
    "Reference ip0 / A": [getattr(cv, "current_scale_target_ip", [None])[0] for cv in phoh_group_scaled],
    "Measured ip0 / A": [getattr(cv, "current_scale_source_ip", [None])[0] for cv in phoh_group_scaled],
})

Current scaling summary:
[Conditions] Exp Type: CV, Solvent: MeCN, Gas: CO₂, Compounds: 0.1 M TBAPF₆, 3 mM Fc, 1 mM Fe-tpyPY2Me, Scan Window: [-1.2, 1], Scan Rate: 100 mV/s, Segments: 3, IR Comp Percent: 100 %


,Compounds,Scale Factor
[0],100 mM PhOH,0.697600
[1],560 mM PhOH,0.739800
[2],1 M PhOH,0.833600
[3],2.8 M PhOH,1.154000


,Label,Scale Factor,Reference ip0 / A,Measured ip0 / A
0,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_100mM...,0.697552,0.00012,0.000172
1,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_560mM...,0.739802,0.00012,0.000162
2,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_1MPhO...,0.833565,0.00012,0.000144
3,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_2.8MP...,1.154291,0.00012,0.000104


## Fit 1: Single Ar CV Seed

Start with one Ar CV at `0.1 V/s`. This gives reasonable starting values before fitting a scan-rate series. The cell block is spelled out here so the defaults are visible; later group fits use `"cell": "auto"` for the same metadata-aware behavior.


In [4]:
single_ar_fit = None
if not HAS_ELECTROKITTY:
    print("SKIP: single Ar fit requires ElectroKitty.")
else:
    AR_FIT_STRIDE = 15
    CO2_FIT_STRIDE = 15

    ar_ee_params = {
        "concentrations": {"bulk": {"FeII": 1.0, "FeI": 0.0, "Fe0": 0.0}},
        "diffusion": {"FeII": 2e-9, "FeI": 2e-9, "Fe0": 2e-9},
        "kinetics": [
            {"E0": -1.3, "k0": 1e-3, "alpha": 0.5},
            {"E0": -1.5, "k0": 1e-3, "alpha": 0.5},
        ],
        "cell": {"T": 298.15, "Ru": 0.0, "Cdl": "auto", "A": 1e-5},
        "spatial": "fast",
    }

    single_ar_fit = e.simulation.fit_cv(
        ar_reference_cv,
        "EE",
        ar_ee_params,
        fit={
            "vary": ["E0_0", "E0_1", "D"],
            "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
            "bounds": "auto",
            "transform": "auto",
        },
        options={
            "cv data": {**AR_CV_WINDOW_OPTIONS, "stride": AR_FIT_STRIDE, "estimate Cdl": "auto"},
            "plot": True,
            "post correction": "offset",
            "print stats": False,
            "print corrections": False,
            "print progress": False,
        },
    )
    single_ar_fit.show({"print setup": False, "print stats": True, "print corrections": True, "print params": False, "print simulation": False})

SKIP: single Ar fit requires ElectroKitty.


## Fit 2: Ar Scan-Rate Group Fit

Fit the Ar scan-rate series to refine the two formal potentials and one tied diffusion coefficient. This cell switches to `"cell": "auto"`, so `Cdl`, area, temperature, and resistance are populated from each source CV or defaults. `Cdl` and `Ru` are per-CV inputs but remain fixed because they are not listed in `fit["vary"]`.


In [5]:
ar_scan_fit = None
if single_ar_fit is None:
    print("SKIP: Ar scan-rate group fit needs the single Ar fit result.")
else:
    ar_scan_params = deepcopy(single_ar_fit.best_params)
    ar_scan_params["cell"] = "auto"

    ar_scan_fit = e.simulation.fit_cvs(
        scan_series,
        "EE",
        ar_scan_params,
        fit={
            "vary": ["E0_0", "E0_1", "D"],
            "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
            "bounds": "auto",
            "transform": "auto",
        },
        per_cv=["cell.Cdl", "cell.Ru"],
        options={
            "cv data": {**AR_CV_WINDOW_OPTIONS, "stride": AR_FIT_STRIDE, "estimate Cdl": "auto"},
            "plot": True,
            "post correction": "offset",
            "print setup": True,
            "print stats": True,
            "print corrections": True,
            "print params": True,
            "print progress": False,
            "print simulation": False,
        },
    )

SKIP: Ar scan-rate group fit needs the single Ar fit result.


## Build The EEC' Concentration-Fit Starting Point

Start from the Ar group-fit parameters. Add substrate/product species and use the Ar-fitted diffusion scale for them unless a better value is known.


In [6]:
group_params = None
if ar_scan_fit is None:
    print("SKIP: EEC' concentration setup needs the Ar scan-rate group fit result.")
else:
    group_params = deepcopy(ar_scan_fit.best_params)
    group_params.setdefault("concentrations", {}).setdefault("bulk", {}).update({
        "Substrate": 1.0,
        "Product": 0.0,
    })
    reference_diffusion = next(iter(group_params.get("diffusion", {"FeII": 1e-9}).values()))
    group_params.setdefault("diffusion", {}).update({
        "Substrate": reference_diffusion,
        "Product": reference_diffusion,
    })
    group_params["reactions"] = [{"kf": 1.0, "kb": 0.0}]
    group_params["cell"] = "auto"

    eecat_mechanism = (
        "E(1):FeII=FeI\n"
        "E(1):FeI=Fe0\n"
        "C:Fe0+Substrate>FeI+Product"
    )

    e.simulation.simulate_cv(
        e.simulation.cv_data(phoh_28m_cv_scaled, {**PHOH_CV_WINDOW_OPTIONS, "stride": CO2_FIT_STRIDE, "estimate Cdl": "auto"}),
        eecat_mechanism,
        group_params,
        options={"plot": False, "check params": True},
    ).show({"print setup": True, "print params": "compact"})

SKIP: EEC' concentration setup needs the Ar scan-rate group fit result.


## Fit 3: CO2/PhOH Concentration Group Fit

Fit the scaled PhOH concentration series. The group setup table should show PhOH mapped to `Substrate`, while `Per-CV Fitting Params` shows the actual per-dataset concentration values in mol/m^3. The scaled CV copies enter `fit_cvs()`, so the fitting algorithm only sees Fc-standardized currents in the trimmed simulation window.


In [7]:
ecat_group_fit = None
if group_params is None:
    print("SKIP: CO2/PhOH concentration group fit needs the EEC' starting params.")
else:
    ecat_group_fit = e.simulation.fit_cvs(
        phoh_group_scaled,
        eecat_mechanism,
        group_params,
        fit={
            "vary": ["reactions.0.kf"],
            "fixed": {"alpha_*": 0.5, "k0_*": 1e-3},
            "bounds": "auto",
            "transform": "auto",
        },
        per_cv=["cell.Cdl", "cell.Ru", "concentrations.bulk.Substrate"],
        options={
            "cv data": {**PHOH_CV_WINDOW_OPTIONS, "stride": CO2_FIT_STRIDE, "estimate Cdl": "auto"},
            "concentration mapping": {"PhOH": "Substrate"},
            "plot": True,
            "post correction": "offset",
            "print setup": True,
            "print stats": True,
            "print corrections": True,
            "print params": True,
            "print progress": False,
            "print simulation": False,
        },
    )

SKIP: CO2/PhOH concentration group fit needs the EEC' starting params.


## Reporting Notes

For group fits, report which parameters are shared, which paths are per-CV, the concentration mapping, the residual correction mode, and whether fixed per-CV values came from source CV metadata or from explicit parameters.
